In [22]:
%pip install -q groq python-dotenv

import os
import getpass
from groq import Groq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

MODEL_NAME = "llama-3.1-8b-instant"

In [23]:
MODEL_CONFIG = {
    "technical": "You are a Technical Support Expert. Provide rigorous, code-focused, and precise answers.",
    "billing": "You are a Billing Support Expert. Be empathetic, financial-focused, and adhere strictly to company refund policies.",
    "general": "You are a General Support Agent. Provide friendly, helpful, and casual chat.",
    "tool": "You are a Tool-Use Expert."
}

In [24]:
def route_prompt(user_input):
    routing_prompt = f"""
    Classify the following user input into EXACTLY ONE of these categories: [technical, billing, general, tool].
    Return ONLY the category name as a single word, with no punctuation or extra text.
    If the user asks for the current price of Bitcoin, classify it as 'tool'.

    User input: {user_input}
    """

    response = client.chat.completions.create(
        messages=[{"role": "user", "content": routing_prompt}],
        model=MODEL_NAME,
        temperature=0,
        max_tokens=10
    )

    return response.choices[0].message.content.strip().lower()

In [25]:
def mock_fetch_bitcoin_price():
    return "[SYSTEM: API CALL EXECUTED] The current price of Bitcoin is $65,432.10 (Mock Data)."

def process_request(user_input):
    print(f"\nUser Input: \"{user_input}\"")

    category = route_prompt(user_input)

    matched_category = "general"
    for key in MODEL_CONFIG.keys():
        if key in category:
            matched_category = key
            break

    print(f"[Router Classified as: {matched_category.upper()}]")

    if matched_category == "tool":
        return mock_fetch_bitcoin_price()

    system_prompt = MODEL_CONFIG.get(matched_category, MODEL_CONFIG["general"])

    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        model=MODEL_NAME,
        temperature=0.7
    )

    return response.choices[0].message.content

In [26]:
test_queries = [
    "My python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription this month.",
    "Hey, how is your day going?",
    "What is the current price of Bitcoin?"
]

for query in test_queries:
    answer = process_request(query)
    print(f"Expert Response: {answer}")
    print("-" * 50)


User Input: "My python script is throwing an IndexError on line 5."
[Router Classified as: TECHNICAL]
Expert Response: To troubleshoot the issue, you'll need to provide more context, such as:

1. The code snippet that's causing the error.
2. The full error message, including the line number and the exception type (`IndexError`).

Please paste the code and the full error message, and I'll help you identify the cause of the `IndexError` and provide a solution.

However, if you're unable to provide the code or error message, I can offer a general explanation of what causes an `IndexError` in Python.

An `IndexError` occurs when you try to access an element in a sequence (such as a list, tuple, or string) with an index that's out of range. In other words, you're trying to access an element that doesn't exist.

Here's a simple example:
```python
my_list = [1, 2, 3]
print(my_list[3])  # IndexError: list index out of range
```
In this example, the list `my_list` only has 3 elements (at indic